# EDA: Encuestas y NLP

## Caso Práctico - Empresa de Telecomunicaciones
## MBI — Prácticas Aplicadas 2026

---

Este notebook analiza la tabla de encuestas de satisfacción de clientes.

**Limitación importante**: estas encuestas son anónimas y se recogen a nivel de zona-mes, no a nivel de cliente individual. Esto significa que **no podemos saber qué cliente concreto respondió cada encuesta**. Solo podemos usar esta información como una señal del entorno de la zona: si una zona tiene encuestas muy negativas, probablemente los clientes de esa zona estén más insatisfechos.

A pesar de esa limitación, esta fuente es muy valiosa porque captura la **percepción subjetiva** del servicio, algo que las variables numéricas de facturación o calidad de red no recogen.

**Variables disponibles:**
- `puntuacion_general_1a5` → nota general del servicio (1-5)
- `nps_0a10` → probabilidad de recomendar el servicio (0-10)
- `texto_libre` → comentario escrito por el cliente
- `sent_text_latente` → sentimiento del texto (-1 muy negativo, +1 muy positivo)
- `flag_incongruente` → 1 si la puntuación y el texto se contradicen
- `stress_calidad` → nivel de estrés de la red ese mes en esa zona
- `indice_calidad_global` → calidad de la red en esa zona ese mes


## Objetivos

1. Explorar la estructura y calidad del dataset
2. Detectar problemas de calidad (valores fuera de rango, incongruencias)
3. Contrastar hipótesis:
   - **H1**: ¿El sentimiento del texto es más negativo en zonas con peor calidad de red?
   - **H2**: ¿Las zonas con peor NPS y puntuación tienen más churn?
   - **H3**: ¿El `flag_incongruente` es una señal de alarma oculta?
   - **H4**: ¿El sentimiento ha evolucionado con el tiempo?
4. Clasificación simple para validar cuánto aportan estas variables al modelo


## 1. Librerías


In [ ]:
import warnings
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.load import load_encuestas, load_clientes, load_churn
from src.utils.helpers import resumen_df, test_mannwhitney, cramers_v

PAL = {'No Churn': '#4C9BE8', 'Churn': '#E85C4C'}
print('Librerías cargadas')

## 2. Carga de datos


In [ ]:
encuestas = load_encuestas()
clientes  = load_clientes()
churn     = load_churn()

print('\nDatasets cargados correctamente')

---
## 3. Exploración inicial y calidad de datos


In [ ]:
resumen_df(encuestas, 'encuestas_texto.csv')
display(encuestas.head())

In [ ]:
display(encuestas.describe())

### 3.1 Problemas de calidad

Ya en el describe se ven dos anomalías claras que hay que documentar:
- `puntuacion_general_1a5` tiene un máximo de 10 — fuera de escala
- `nps_0a10` tiene un máximo de 24 — fuera de escala

También hay 16 encuestas sin `texto_libre`.


In [ ]:
# Valores fuera de rango
print('Valores de puntuacion_general_1a5:')
print(encuestas['puntuacion_general_1a5'].value_counts().sort_index().to_dict())
print(f'  → Fuera de rango (> 5): {(encuestas["puntuacion_general_1a5"] > 5).sum()} registros')
print()
print('Valores de nps_0a10:')
print(encuestas['nps_0a10'].value_counts().sort_index().to_dict())
print(f'  → Fuera de rango (> 10): {(encuestas["nps_0a10"] > 10).sum()} registros')
print()
print(f'Encuestas sin texto_libre: {encuestas["texto_libre"].isnull().sum()}')
print(f'Encuestas incongruentes (flag=1): {encuestas["flag_incongruente"].sum()} '
      f'({encuestas["flag_incongruente"].mean()*100:.1f}%)')

In [ ]:
# Limpieza: valores fuera de rango → NaN
# (la corrección definitiva está en src/clean.py)
enc = encuestas.copy()
enc.loc[enc['puntuacion_general_1a5'] > 5, 'puntuacion_general_1a5'] = np.nan
enc.loc[enc['nps_0a10'] > 10, 'nps_0a10'] = np.nan

print(f'Puntuaciones corregidas: {(encuestas["puntuacion_general_1a5"] > 5).sum()}')
print(f'NPS corregidos:          {(encuestas["nps_0a10"] > 10).sum()}')
print()
print('Distribución de puntuacion (tras limpieza):')
print(enc['puntuacion_general_1a5'].value_counts().sort_index().to_dict())

### 3.2 Distribución de las variables de satisfacción


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

# Puntuación general
counts_pun = enc['puntuacion_general_1a5'].value_counts().sort_index()
axes[0].bar(counts_pun.index.astype(str), counts_pun.values, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de puntuación general (1-5)', fontweight='bold')
axes[0].set_xlabel('Puntuación')
axes[0].set_ylabel('Nº encuestas')
for bar, val in zip(axes[0].patches, counts_pun.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val}', ha='center', va='bottom', fontsize=9)

# NPS
counts_nps = enc['nps_0a10'].value_counts().sort_index()
axes[1].bar(counts_nps.index.astype(str), counts_nps.values, color='coral', edgecolor='white')
axes[1].set_title('Distribución del NPS (0-10)', fontweight='bold')
axes[1].set_xlabel('NPS')
axes[1].set_ylabel('Nº encuestas')

# Sentimiento del texto
enc['sent_text_latente'].hist(bins=30, ax=axes[2], color='#9b59b6', edgecolor='white', alpha=0.8)
axes[2].axvline(0, color='black', linestyle='--', linewidth=1, label='Neutro')
axes[2].axvline(enc['sent_text_latente'].mean(), color='red', linestyle='--',
                label=f'Media: {enc["sent_text_latente"].mean():.2f}')
axes[2].set_title('Distribución del sentimiento del texto', fontweight='bold')
axes[2].set_xlabel('Sentimiento (-1 negativo, +1 positivo)')
axes[2].set_ylabel('Frecuencia')
axes[2].legend(fontsize=9)

# Stress de calidad
enc['stress_calidad'].hist(bins=25, ax=axes[3], color='#f39c12', edgecolor='white', alpha=0.8)
axes[3].axvline(enc['stress_calidad'].mean(), color='red', linestyle='--',
                label=f'Media: {enc["stress_calidad"].mean():.2f}')
axes[3].set_title('Distribución del stress de calidad de red', fontweight='bold')
axes[3].set_xlabel('Stress (0 = buena red, 1 = red muy saturada)')
axes[3].set_ylabel('Frecuencia')
axes[3].legend(fontsize=9)

plt.suptitle('Distribución de variables de encuestas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Sentimiento positivo (> 0): {(enc['sent_text_latente'] > 0).mean()*100:.1f}%")
print(f"Sentimiento negativo (< 0): {(enc['sent_text_latente'] < 0).mean()*100:.1f}%")
print(f"NPS medio: {enc['nps_0a10'].mean():.1f} / 10")
print(f"Puntuación media: {enc['puntuacion_general_1a5'].mean():.1f} / 5")

### 3.3 Vistazo al texto libre

Antes de entrar en hipótesis, vale la pena leer algunos ejemplos del texto libre para entender qué dicen los clientes.


In [ ]:
# Ejemplos de texto muy negativo
print('=== TEXTOS MÁS NEGATIVOS (sent_text_latente < -0.9) ===')
muy_neg = enc[enc['sent_text_latente'] < -0.9]['texto_libre'].dropna().head(5)
for i, t in enumerate(muy_neg, 1):
    print(f'{i}. {t}')

print()
print('=== TEXTOS MÁS POSITIVOS (sent_text_latente > 0.3) ===')
muy_pos = enc[enc['sent_text_latente'] > 0.3]['texto_libre'].dropna().head(5)
for i, t in enumerate(muy_pos, 1):
    print(f'{i}. {t}')

print()
print('=== EJEMPLOS INCONGRUENTES (puntuacion alta, texto negativo) ===')
incon = enc[(enc['flag_incongruente'] == 1) &
            (enc['puntuacion_general_1a5'] >= 4)][['puntuacion_general_1a5',
                                                    'sent_text_latente',
                                                    'texto_libre']].head(3)
display(incon)

### 3.4 Palabras más frecuentes en los textos

Un análisis sencillo de frecuencia de palabras para entender qué temas aparecen más en las quejas.


In [ ]:
# Palabras de parada en español (stopwords básicas)
STOPWORDS = {
    'de', 'la', 'el', 'en', 'y', 'a', 'los', 'las', 'un', 'una', 'que',
    'es', 'se', 'no', 'con', 'por', 'para', 'del', 'al', 'lo', 'como',
    'más', 'pero', 'si', 'su', 'le', 'ya', 'o', 'este', 'me', 'mi',
    'muy', 'ha', 'hay', 'he', 'ni', 'mi', 'desde', 'todo', 'hace',
    'tengo', 'cuando', 'sobre', 'hasta', 'sin', 'nos', 'les', 'sus'
}

def contar_palabras(textos, stopwords=STOPWORDS):
    contador = Counter()
    for texto in textos:
        if pd.isna(texto):
            continue
        palabras = texto.lower().replace('.', '').replace(',', '').split()
        contador.update([p for p in palabras if p not in stopwords and len(p) > 3])
    return contador

# Top palabras en textos negativos vs positivos
neg = enc[enc['sent_text_latente'] < -0.3]['texto_libre']
pos = enc[enc['sent_text_latente'] > 0.0]['texto_libre']

top_neg = pd.Series(dict(contar_palabras(neg).most_common(15)))
top_pos = pd.Series(dict(contar_palabras(pos).most_common(15)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_neg.sort_values().plot(kind='barh', ax=axes[0], color='#E85C4C', alpha=0.8)
axes[0].set_title('Palabras más frecuentes — Textos negativos', fontweight='bold')
axes[0].set_xlabel('Frecuencia')

top_pos.sort_values().plot(kind='barh', ax=axes[1], color='#4C9BE8', alpha=0.8)
axes[1].set_title('Palabras más frecuentes — Textos positivos', fontweight='bold')
axes[1].set_xlabel('Frecuencia')

plt.suptitle('Análisis de frecuencia de palabras en el texto libre',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Análisis de Hipótesis


### H1 — ¿El sentimiento es más negativo en zonas con peor calidad de red?

Si los clientes de zonas con mala red escriben textos más negativos, eso confirmaría que el sentimiento de las encuestas está capturando la experiencia real de calidad. Y tendría sentido usarlo como variable en el modelo.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sentimiento por tipo de zona
orden_zonas = ['rural', 'suburbana', 'urbana_premium']
enc_zonas = enc[enc['tipo_zona'].isin(orden_zonas)]
sns.boxplot(data=enc_zonas, x='tipo_zona', y='sent_text_latente',
            order=orden_zonas,
            palette=sns.color_palette('RdYlGn', 3), ax=axes[0])
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Sentimiento por tipo de zona', fontweight='bold')
axes[0].set_xlabel('Tipo de zona')
axes[0].set_ylabel('Sentimiento del texto')

# Correlación entre stress de red y sentimiento
axes[1].scatter(enc['stress_calidad'], enc['sent_text_latente'],
                alpha=0.3, color='steelblue', s=15)
# Línea de tendencia
z = np.polyfit(enc['stress_calidad'].dropna(),
               enc.loc[enc['stress_calidad'].notna(), 'sent_text_latente'], 1)
p = np.poly1d(z)
x_line = np.linspace(enc['stress_calidad'].min(), enc['stress_calidad'].max(), 100)
axes[1].plot(x_line, p(x_line), color='red', linewidth=2, label='Tendencia')
axes[1].set_title('Stress de red vs Sentimiento del texto', fontweight='bold')
axes[1].set_xlabel('Stress de calidad (0=buena red, 1=mala red)')
axes[1].set_ylabel('Sentimiento')
axes[1].legend()

# Sentimiento por tramos de calidad global
enc['tramo_calidad'] = pd.qcut(enc['indice_calidad_global'], q=3,
                                labels=['Baja calidad', 'Media', 'Alta calidad'])
sns.boxplot(data=enc, x='tramo_calidad', y='sent_text_latente',
            order=['Baja calidad', 'Media', 'Alta calidad'],
            palette=sns.color_palette('RdYlGn', 3), ax=axes[2])
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title('Sentimiento por tramo de calidad global', fontweight='bold')
axes[2].set_xlabel('Tramo de calidad de red')
axes[2].set_ylabel('Sentimiento del texto')

plt.suptitle('H1 — Sentimiento vs Calidad de red', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlación de Pearson entre stress y sentimiento
corr = enc[['stress_calidad', 'indice_calidad_global', 'sent_text_latente',
            'puntuacion_general_1a5', 'nps_0a10']].corr().round(3)
print('Correlaciones con sent_text_latente:')
print(corr['sent_text_latente'].drop('sent_text_latente'))

### Resultado H1

La hipótesis se confirma con claridad. Hay una **relación directa entre la calidad de red y el sentimiento** de los textos:

- Las zonas **rurales tienen el sentimiento más negativo** (mediana ~-0.75), seguidas de suburbana (~-0.55) y urbana_premium (~-0.40). La diferencia entre rural y urbana_premium es notable.
- El scatter de stress vs sentimiento muestra una **tendencia descendente clara** (r = -0.41): a más estrés de red, más negativo el texto. La línea roja lo confirma visualmente.
- El boxplot por tramos de calidad global refuerza lo mismo: zonas con baja calidad tienen sentimiento mediano de ~-0.65, mientras que zonas con alta calidad llegan a ~-0.35.

Las correlaciones lo confirman numéricamente:
- `stress_calidad` → r = **-0.41** con el sentimiento
- `indice_calidad_global` → r = **+0.41** con el sentimiento
- `nps_0a10` → r = **+0.49** con el sentimiento (el más alto)

Esto valida que el `sent_text_latente` no es ruido — está capturando la experiencia real de la red. Un cliente que sufre mala cobertura lo expresa en el texto aunque le dé una puntuación numérica moderada.


### H2 — ¿Las zonas con peor NPS y puntuación tienen más churn?

Esta es la hipótesis clave para justificar el uso de las encuestas en el modelo. Si las zonas donde los clientes puntúan peor el servicio tienen más churn, entonces las métricas de las encuestas son predictores válidos aunque no estén a nivel de cliente individual.

Para analizarlo, calculamos la media de NPS, puntuación y sentimiento por zona, y la cruzamos con la tasa de churn de los clientes de esa zona.


In [ ]:
# Métricas de encuestas agregadas por zona
enc_zona = enc.groupby('zona_id').agg(
    nps_medio          = ('nps_0a10',              'mean'),
    puntuacion_media   = ('puntuacion_general_1a5', 'mean'),
    sentimiento_medio  = ('sent_text_latente',      'mean'),
    pct_incongruentes  = ('flag_incongruente',      'mean'),
    stress_medio       = ('stress_calidad',         'mean'),
    n_encuestas        = ('encuesta_id',            'count'),
).reset_index()

# Tasa de churn por zona (a través de clientes)
churn_agg = churn.groupby('cliente_id')['churn'].max().reset_index()
churn_agg.columns = ['cliente_id', 'ever_churn']

churn_zona = (clientes[['cliente_id', 'zona_id']]
              .merge(churn_agg, on='cliente_id', how='inner')
              .groupby('zona_id')['ever_churn']
              .agg(['mean', 'count'])
              .reset_index()
              .rename(columns={'mean': 'tasa_churn', 'count': 'n_clientes'}))

# Tabla analítica zona: encuestas + churn
df_zona = enc_zona.merge(churn_zona, on='zona_id', how='inner')

print(f'Zonas en el análisis: {len(df_zona)}')
display(df_zona.head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

vars_enc = [
    ('nps_medio',         'NPS medio de la zona'),
    ('puntuacion_media',  'Puntuación media de la zona'),
    ('sentimiento_medio', 'Sentimiento medio de la zona'),
]

for i, (var, titulo) in enumerate(vars_enc):
    axes[i].scatter(df_zona[var], df_zona['tasa_churn'] * 100,
                    s=df_zona['n_clientes'] / 5,  # tamaño = nº clientes de la zona
                    alpha=0.7, color='steelblue', edgecolors='white')
    # Línea de tendencia
    datos = df_zona[[var, 'tasa_churn']].dropna()
    z = np.polyfit(datos[var], datos['tasa_churn'] * 100, 1)
    p = np.poly1d(z)
    x_line = np.linspace(datos[var].min(), datos[var].max(), 100)
    axes[i].plot(x_line, p(x_line), color='red', linewidth=2, linestyle='--')
    axes[i].set_title(f'Tasa de churn vs {titulo}', fontweight='bold')
    axes[i].set_xlabel(titulo)
    axes[i].set_ylabel('Tasa de churn (%) por zona')
    # Correlación
    corr_val = datos[var].corr(datos['tasa_churn'])
    axes[i].text(0.05, 0.95, f'r = {corr_val:.2f}',
                 transform=axes[i].transAxes, fontsize=10,
                 verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('H2 — Satisfacción por zona vs Tasa de churn',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlaciones con tasa_churn por zona:')
for var, _ in vars_enc:
    r = df_zona[var].corr(df_zona['tasa_churn'])
    print(f'  {var:25s}: r = {r:.3f}')

### Resultado H2

Este es el resultado más potente de todo el notebook. Las correlaciones entre las métricas de encuestas por zona y la tasa de churn son **muy altas**:

- `nps_medio` → r = **-0.79** con tasa de churn
- `sentimiento_medio` → r = **-0.78** con tasa de churn
- `puntuacion_media` → r = **-0.65** con tasa de churn

Los tres scatter plots muestran una tendencia descendente muy clara y consistente: **cuanto peor valoran los clientes el servicio en una zona, más abandonan**. La zona Z01 por ejemplo tiene NPS de 1.47, puntuación de 2.80, sentimiento de -0.62 y una tasa de churn del 34%. La zona Z05 tiene NPS de 3.53, puntuación de 3.42, sentimiento de -0.28 y solo el 14% de churn.

Correlaciones de r = -0.79 son inusuales en ciencia de datos. Esto significa que las encuestas de satisfacción son un **predictor muy fuerte del churn a nivel de zona**, aunque no podamos asignarlas a clientes individuales.

La limitación sigue siendo la misma: trabajamos a nivel de zona, no de cliente. Pero la señal es tan clara que justifica incluir estas variables en el modelo final.


### H3 — ¿El `flag_incongruente` es una señal de alarma oculta?

El `flag_incongruente` marca encuestas donde el cliente pone una nota alta pero el texto es negativo. Esto puede interpretarse como un cliente que no quiere quejarse abiertamente pero que en realidad está insatisfecho. Si las zonas con más incongruencias tienen más churn, sería una señal de alarma silenciosa muy interesante.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Comparar sentimiento y puntuación entre incongruentes y congruentes
enc['incongruente_label'] = enc['flag_incongruente'].map(
    {0: 'Congruente', 1: 'Incongruente'}
)

sns.boxplot(data=enc, x='incongruente_label', y='sent_text_latente',
            order=['Congruente', 'Incongruente'],
            palette={'Congruente': '#4C9BE8', 'Incongruente': '#E85C4C'},
            ax=axes[0])
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Sentimiento del texto:\nCongruentes vs Incongruentes', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Sentimiento (-1 negativo, +1 positivo)')

# % incongruentes por zona vs tasa de churn
axes[1].scatter(df_zona['pct_incongruentes'], df_zona['tasa_churn'] * 100,
                s=df_zona['n_clientes'] / 5, alpha=0.7,
                color='#E85C4C', edgecolors='white')
datos_incon = df_zona[['pct_incongruentes', 'tasa_churn']].dropna()
z = np.polyfit(datos_incon['pct_incongruentes'], datos_incon['tasa_churn'] * 100, 1)
p = np.poly1d(z)
x_line = np.linspace(datos_incon['pct_incongruentes'].min(),
                     datos_incon['pct_incongruentes'].max(), 100)
axes[1].plot(x_line, p(x_line), color='black', linewidth=2, linestyle='--')
r_incon = datos_incon['pct_incongruentes'].corr(datos_incon['tasa_churn'])
axes[1].set_title('% encuestas incongruentes por zona\nvs tasa de churn', fontweight='bold')
axes[1].set_xlabel('% encuestas incongruentes')
axes[1].set_ylabel('Tasa de churn (%) por zona')
axes[1].text(0.05, 0.95, f'r = {r_incon:.2f}',
             transform=axes[1].transAxes, fontsize=10,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('H3 — El flag_incongruente como señal de alarma',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Sentimiento medio:')
print(enc.groupby('incongruente_label')['sent_text_latente'].mean().round(3))
print()
print('Puntuación media:')
print(enc.groupby('incongruente_label')['puntuacion_general_1a5'].mean().round(3))

### Resultado H3

La hipótesis **no se confirma**. El resultado es sorprendente:

- El sentimiento de los incongruentes (-0.47) es prácticamente igual al de los congruentes (-0.51). No hay diferencia real en el texto.
- La puntuación numérica sí difiere mucho: incongruentes tienen media de **3.93** frente a **2.95** de los congruentes. Eso tiene sentido porque el flag se define precisamente por tener nota alta con texto negativo.
- La correlación entre % de incongruentes por zona y tasa de churn es de **r = -0.01**, prácticamente cero. Las zonas con más incongruentes no tienen más ni menos churn.

¿Por qué no funciona como señal de alarma? Una explicación es que el `flag_incongruente` está mal calibrado en el dataset sintético, o que la incongruencia no está capturando insatisfacción oculta sino simplemente clientes que responden la puntuación numérica de forma positiva por costumbre.

**Conclusión práctica**: el `flag_incongruente` no aporta valor predictivo y no lo incluiremos en el modelo final.


### H4 — ¿El sentimiento ha evolucionado con el tiempo?

Como el 5G ha mejorado y la calidad de red en general ha subido, es de esperar que el sentimiento de los clientes también haya mejorado. Si vemos esa tendencia, confirma que las encuestas están midiendo algo real y que las mejoras de red se perciben.


In [ ]:
evol = enc.groupby('fecha').agg(
    sentimiento_medio  = ('sent_text_latente',      'mean'),
    nps_medio          = ('nps_0a10',               'mean'),
    puntuacion_media   = ('puntuacion_general_1a5',  'mean'),
    pct_incongruentes  = ('flag_incongruente',       'mean'),
    stress_medio       = ('stress_calidad',          'mean'),
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Sentimiento
axes[0].plot(evol['fecha'], evol['sentimiento_medio'],
             color='#9b59b6', linewidth=2)
axes[0].fill_between(evol['fecha'], evol['sentimiento_medio'],
                     alpha=0.15, color='#9b59b6')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1, label='Neutro')
axes[0].set_title('Evolución del sentimiento medio', fontweight='bold')
axes[0].set_ylabel('Sentimiento')
axes[0].legend(fontsize=9)

# NPS y puntuación
ax2b = axes[1].twinx()
axes[1].plot(evol['fecha'], evol['nps_medio'],
             color='coral', linewidth=2, label='NPS medio')
ax2b.plot(evol['fecha'], evol['puntuacion_media'],
          color='steelblue', linewidth=2, linestyle='--', label='Puntuación media')
axes[1].set_title('Evolución del NPS y puntuación media', fontweight='bold')
axes[1].set_ylabel('NPS (0-10)', color='coral')
ax2b.set_ylabel('Puntuación (1-5)', color='steelblue')
axes[1].legend(loc='upper left', fontsize=9)
ax2b.legend(loc='upper right', fontsize=9)

# % incongruentes
axes[2].plot(evol['fecha'], evol['pct_incongruentes'] * 100,
             color='#e67e22', linewidth=2)
axes[2].fill_between(evol['fecha'], evol['pct_incongruentes'] * 100,
                     alpha=0.15, color='#e67e22')
axes[2].set_title('Evolución del % de encuestas incongruentes', fontweight='bold')
axes[2].set_ylabel('% incongruentes')
axes[2].set_xlabel('Fecha')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('H4 — Evolución temporal de las métricas de encuestas',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Resultado H4

Se confirma una mejora gradual pero con mucho ruido mes a mes:

- El **sentimiento mejora progresivamente**: empieza en ~-0.60 en 2023 y sube hasta ~-0.40 a finales de 2025. La dirección es la esperada — las mejoras de red se perciben — aunque nunca llega a terreno positivo. El servicio sigue siendo valorado negativamente en conjunto.
- El **NPS y la puntuación también mejoran** suavemente, aunque con mucha variabilidad mensual. El NPS sube de ~2.3 a ~3.5 y la puntuación de ~3.2 a ~3.5 aproximadamente.
- El **% de incongruentes no muestra tendencia clara**: oscila entre el 10% y el 45% sin patrón definido, lo que refuerza la conclusión de H3 de que no es una señal útil.

El mensaje para la presentación es claro: **la empresa ha mejorado su red y eso se refleja en la percepción de los clientes**, pero el nivel de satisfacción sigue siendo bajo. Un NPS medio de 2-3 sobre 10 en 2025 sigue siendo territorio de detractores.


---
## 5. Clasificación simple — Validación cuantitativa

Entrenamos un modelo simple usando solo las variables de encuestas agregadas por zona para ver cuánto aportan a predecir el churn.

⚠️ Recuerda que aquí trabajamos a nivel de zona, no de cliente. Cada cliente hereda las métricas de encuestas de su zona.


In [ ]:
# Tabla analítica: cliente + encuestas de su zona + ever_churn
df_modelo = (clientes[['cliente_id', 'zona_id']]
             .merge(churn_agg,  on='cliente_id', how='inner')
             .merge(enc_zona,   on='zona_id',    how='left'))

vars_modelo = ['nps_medio', 'puntuacion_media', 'sentimiento_medio',
               'pct_incongruentes', 'stress_medio']

df_modelo = df_modelo.dropna(subset=vars_modelo + ['ever_churn'])
X = df_modelo[vars_modelo]
y = df_modelo['ever_churn']

print(f'Clientes para el modelo: {len(X):,}')
print(f'Tasa de churn: {y.mean()*100:.1f}%\n')

# Regresión logística
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(random_state=42, max_iter=1000))
])
auc_lr = cross_val_score(pipe_lr, X, y, cv=5, scoring='roc_auc').mean()

# Árbol de decisión
pipe_dt = Pipeline([
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42))
])
auc_dt = cross_val_score(pipe_dt, X, y, cv=5, scoring='roc_auc').mean()

print(f'AUC Regresión Logística: {auc_lr:.3f}')
print(f'AUC Árbol de Decisión:   {auc_dt:.3f}')

# Importancia de variables
pipe_dt.fit(X, y)
importancias = pd.Series(
    pipe_dt.named_steps['model'].feature_importances_,
    index=vars_modelo
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importancias.plot(kind='barh', ax=ax, color='#9b59b6', alpha=0.8)
ax.set_title('Importancia de variables — Árbol de decisión\n(solo variables de encuestas)',
             fontweight='bold')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

### Resultado clasificación

- **AUC Regresión Logística: 0.599**
- **AUC Árbol de Decisión: 0.596**

El AUC es prácticamente idéntico al que obtuvimos con las variables de calidad de red (~0.60). Esto no es casualidad: ambas fuentes están midiendo aspectos relacionados con la red, y el árbol lo confirma — **`stress_medio` acapara casi toda la importancia (~95%)**, mientras que `nps_medio`, `puntuacion_media`, `pct_incongruentes` y `sentimiento_medio` apenas contribuyen.

¿Qué significa esto? Que cuando el árbol tiene acceso al `stress_calidad` (que ya teníamos en facturación y soporte), el resto de variables de encuesta no añaden información nueva. Las encuestas y la calidad de red están correlacionadas porque miden lo mismo desde ángulos distintos.

Sin embargo, esto **no significa que las encuestas sean inútiles**. La correlación r = -0.79 de H2 es muy potente y puede añadir valor en el modelo final cuando se combine con variables de comportamiento (impagos, interacciones de soporte). El problema aquí es que solo usamos variables de encuesta, sin el resto de fuentes.


---
## 6. Conclusiones del EDA de Encuestas

**Distribución general**: El NPS medio es de 2.7/10 y el sentimiento medio del texto es -0.50. El 91% de los textos son negativos. La empresa tiene un problema serio de percepción del servicio, aunque ha mejorado gradualmente desde 2023.

**Palabras más frecuentes en textos negativos**: cobertura, zona, servicio, congestión, velocidad baja. Todo apunta a problemas de red, no de atención al cliente.

**H1 — Confirmada**: El sentimiento es más negativo en zonas con peor red (r = -0.41 con stress). Las encuestas están capturando la experiencia real de calidad de red.

**H2 — Confirmada con fuerza**: Las zonas con peor NPS y sentimiento tienen más churn (r = -0.79 y r = -0.78). Es la correlación más alta que hemos visto en todo el proyecto.

**H3 — No confirmada**: El flag_incongruente no predice el churn (r = -0.01). No lo incluimos en el modelo.

**H4 — Confirmada parcialmente**: El sentimiento mejora con el tiempo pero sigue siendo negativo. Las mejoras de red se perciben pero no son suficientes.

**Clasificación**: AUC ~0.60, dominado por `stress_medio`. Las variables de texto (sentimiento, NPS) no añaden poder predictivo cuando el stress ya está disponible.

---

### Variables recomendadas para el modelo final
- `nps_medio` de la zona → correlación -0.79 con churn por zona, muy potente
- `sentimiento_medio` de la zona → correlación -0.78, captura percepción subjetiva
- `puntuacion_media` de la zona → correlación -0.65, más ruidosa pero complementaria
- `stress_calidad` → ya la teníamos, aquí se confirma su importancia
- **Descartar**: `flag_incongruente` → sin poder predictivo

⚠️ **Para la defensa**: estas variables son a nivel de zona, no de cliente. Dos clientes en la misma zona reciben el mismo valor. Es una aproximación válida pero hay que mencionarla siempre.

---
*Notebook elaborado como parte de la asignatura de Prácticas Aplicadas — Máster en Ciencia de Datos 2026*
